# A2 — Representations: definitions, permutation rep of S₃, regular rep

**What this notebook claims** (state without looking at phase exit):

1. A representation is a homomorphism $\rho: G \to GL(V)$; equivalence, unitarity, faithfulness, reducibility/irreducibility.
2. S₃ acts on $\mathbb{R}^3$ by permuting basis vectors (**permutation representation**). It is reducible: $\mathbb{R}^3 = \text{trivial} \oplus \text{standard}$ (a 1-dim invariant line and an invariant plane), and the standard rep is irreducible.
3. A rep is irreducible iff $\langle \chi_\rho, \chi_\rho \rangle = 1$ (inner product over the group with class weighting).
4. The **regular representation** (G acting on its own group algebra) decomposes as $\bigoplus_i d_i\,\rho_i$: every irrep appears, with multiplicity its dimension. For S₃: $\text{reg} = 1 \cdot \text{trivial} \oplus 1 \cdot \text{sign} \oplus 2 \cdot \text{standard}$ (dimensions 6 = 1+1+4).

Source map: Woit ch. 2, Zee part I. *Verify section numbers against your copy.*

## Definitions

A **representation** of $G$ on a vector space $V$ is a map $\rho: G \to GL(V)$ with $\rho(gh) = \rho(g)\rho(h)$, $\rho(e) = I$.
It is **irreducible** if $V$ has no nontrivial invariant subspace. **Character**: $\chi_\rho(g) = \mathrm{tr}\,\rho(g)$ — constant on conjugacy classes.
Inner product: $\langle \chi, \psi\rangle = \frac{1}{|G|}\sum_g \overline{\chi(g)}\,\psi(g)$. Claim 3 says $\langle\chi_\rho,\chi_\rho\rangle = 1 \iff \rho$ irreducible.

In [ ]:
import sympy as sp

# S3 as in A1
def compose(p, q):
    return tuple(p[q[i]] for i in range(3))
def inverse(p):
    inv = [0]*3
    for i, j in enumerate(p):
        inv[j] = i
    return tuple(inv)
from itertools import permutations
S3 = list(permutations(range(3)))
e = (0, 1, 2)

# permutation representation on R^3: rho(g) maps basis vector i -> basis vector g(i)
def perm_matrix(g):
    M = sp.zeros(3)
    for i in range(3):
        M[g[i], i] = 1
    return M
rho = {g: perm_matrix(g) for g in S3}

# Claim 1: homomorphism property, checked on every pair
assert all(rho[compose(g, h)] == rho[g] * rho[h] for g in S3 for h in S3), "not a homomorphism"
assert rho[e] == sp.eye(3)
print("rho: S3 -> GL(3) is a representation (homomorphism property verified on all 36 pairs)")

## Claim 2: decomposition of the permutation representation

The vector $v = (1,1,1)$ is fixed by every permutation $\Rightarrow$ span$(v)$ is a 1-dim invariant subspace carrying the **trivial** rep.
Its orthogonal complement (the plane $x+y+z=0$) is invariant too, carrying the **standard** rep.
We exhibit both invariant subspaces, then prove the 2-dim piece is irreducible via the character test (Claim 3).

In [ ]:
v = sp.Matrix([1, 1, 1])
# invariant line: every perm matrix fixes v
assert all(rho[g] * v == v for g in S3), "line not invariant"
# invariant complement: plane perpendicular to v
basis_plane = [sp.Matrix([1, -1, 0]), sp.Matrix([0, 1, -1])]
assert all((rho[g] * w).dot(v) == 0 for g in S3 for w in basis_plane), "plane not invariant"

def restrict(g, basis):
    # invariant subspace: rho(g) B = B A_g  =>  A_g = (B^T B)^{-1} B^T rho(g) B
    B = sp.Matrix.hstack(*basis)
    return sp.simplify((B.T * B).inv() * B.T * rho[g] * B)
std = {g: restrict(g, basis_plane) for g in S3}
# check the restriction is again a homomorphism into GL(2)
assert all(std[compose(g, h)] == std[g] * std[h] for g in S3 for h in S3)
print("R^3 = span(1,1,1) [trivial] + plane [standard]; standard rep matrices in plane basis:")
for g in S3:
    print(f"  g={g}: {list(std[g])}")

## Claim 3: the irreducibility criterion, applied

$\langle \chi_\rho, \chi_\rho \rangle = \frac{1}{|G|} \sum_g |\chi(g)|^2$ counts the number of irreducible constituents (with multiplicity), so it equals 1 exactly for irreducibles.
We compute each character **from its matrix representation**, never from a table.

In [ ]:
def char(rep, g):
    return sp.trace(rep[g])

def inner(chars1, chars2):
    return sp.Rational(sum(chars1[g] * chars2[g] for g in S3), len(S3))   # all chars real here

chi_perm = {g: char(rho, g) for g in S3}
chi_triv = {g: 1 for g in S3}
chi_sign = {g: sp.sign(sp.det(rho[g])) for g in S3}              # sign = determinant of perm matrix
chi_std  = {g: char(std, g) for g in S3}

for name, chi in [("trivial", chi_triv), ("sign", chi_sign), ("standard", chi_std), ("perm", chi_perm)]:
    print(f"<{name},{name}> = {inner(chi, chi)}")

assert inner(chi_triv, chi_triv) == 1 and inner(chi_sign, chi_sign) == 1
assert inner(chi_std, chi_std) == 1, "standard rep must be irreducible"
assert inner(chi_perm, chi_perm) == 2, "perm rep has 2 constituents"
# cross-check: perm = trivial + standard, as characters
assert all(chi_perm[g] == chi_triv[g] + chi_std[g] for g in S3)
print("standard is irreducible; perm = trivial + standard (verified at the character level)")

## Claim 4: the regular representation

Let $G$ act on the vector space with basis $\{|g\rangle\}$ by $\rho_{reg}(h)|g\rangle = |hg\rangle$.
Decomposition theorem: $\rho_{reg} \cong \bigoplus_i d_i \rho_i$.
We build it for S₃ and extract multiplicities by projecting its character against trivial / sign / standard — each projection formula itself derived from orthogonality.

In [ ]:
# regular rep: 6x6 matrices, basis labelled by group elements
idx = {g: i for i, g in enumerate(S3)}
reg = {}
for h in S3:
    M = sp.zeros(6)
    for g in S3:
        M[idx[compose(h, g)], idx[g]] = 1
    reg[h] = M
chi_reg = {g: sp.trace(reg[g]) for g in S3}
print("chi_reg(e) =", chi_reg[e], "= |S3|; chi_reg(g!=e) =", chi_reg[S3[1]])

# multiplicity of irrep i in reg = <chi_reg, chi_i>  (projection formula)
mult = {}
for name, chi in [("trivial", chi_triv), ("sign", chi_sign), ("standard", chi_std)]:
    mult[name] = inner(chi_reg, chi)
print("multiplicities in regular rep:", {k: int(v) for k, v in mult.items()})
assert mult["trivial"] == 1 and mult["sign"] == 1 and mult["standard"] == 2
# dimension check: |G| = sum d_i^2  (here: 6 = 1 + 1 + 4)
assert sum(int(v)**2 for v in mult.values()) == 6
print("reg = 1*trivial + 1*sign + 2*standard;  sum d_i^2 = 6 = |S3|  -> Claim 4 verified")

## Physics note

A representation is exactly how a symmetry group acts on the Hilbert space of a quantum system; an **irrep** is a multiplet of states that symmetry maps among themselves but never mixes with outsiders — hence degenerate energy levels come in irrep-sized bunches. Zee's opening chapters hammer this identification; Woit ch. 2 states it precisely. (Fill in the precise statements in *My summary*.)

## My summary

*(What is a representation, in one sentence a physicist would love? Why does $\langle\chi,\chi\rangle=1$ mean irreducible?)*

-


## Gaps

- 